# M3L2 E02 - Chat con memoria

## Objetivo

Un LLM no recuerda conversaciones anteriores por si solo. Cada llamada es independiente. En este ejercicio vas a guardar historial por `session_id`.

## Memoria vs contexto

| Concepto | Que significa |
|---|---|
| Contexto | Informacion enviada en la llamada actual |
| Historial | Mensajes anteriores que volvemos a enviar |
| Memoria | Mecanismo que guarda y recupera historial |
| `session_id` | Identificador de una conversacion |

## Diagrama

```text
Usuario pregunta -> buscar historial -> prompt con historial -> modelo -> guardar respuesta
```

In [ ]:
# !pip install langchain langchain-openai

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")


def obtener_modelo(temperature: float = 0.2):
    return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)


print("API key cargada en la variable de entorno OPENAI_API_KEY")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = obtener_modelo()
parser = StrOutputParser()
historiales = {}

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente claro y breve. Usa el historial si ayuda a responder."),
    MessagesPlaceholder("historial"),
    ("human", "{pregunta}"),
])
cadena = prompt | llm | parser

In [ ]:
def obtener_historial(session_id: str):
    if session_id not in historiales:
        historiales[session_id] = InMemoryChatMessageHistory()
    return historiales[session_id]

In [ ]:
chat_con_memoria = RunnableWithMessageHistory(
    cadena,
    obtener_historial,
    input_messages_key="pregunta",
    history_messages_key="historial",
)

In [ ]:
config = {"configurable": {"session_id": "alumno-1"}}
respuesta_1 = chat_con_memoria.invoke({"pregunta": "Me llamo Ana y estoy aprendiendo LangChain."}, config=config)
respuesta_2 = chat_con_memoria.invoke({"pregunta": "Como me llamo y que estoy aprendiendo?"}, config=config)
print("Respuesta 1:", respuesta_1)
print("Respuesta 2:", respuesta_2)

## Errores comunes

| Error | Causa |
|---|---|
| El modelo no recuerda | No se uso el mismo `session_id` |
| Falla el placeholder | El nombre no coincide con `history_messages_key` |
| Todos comparten memoria | Se usa siempre el mismo id |

In [ ]:
assert chat_con_memoria is not None
assert "alumno-1" in historiales
print("Checks OK")

## Resumen

La memoria no es magia: es historial guardado y reenviado al modelo.